After the cleaning and exploration, we will start with developing a baseline:

In [0]:
import pyspark.pandas as ps
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.dates as mdates
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import scipy.stats as ss
import xgboost as xgb
import shap
import os
import joblib

In [0]:

# load tables
activiteiten = spark.table("default.cleaned_activiteiten")
ozp = spark.table("default.cleaned_ozp")
verblijf = spark.table("default.cleaned_verblijf")
productiviteit = spark.table("default.cleaned_productiviteit")
diensten = spark.table("default.cleaned_diensten")
tarieven = spark.table("default.cleaned_tarieven")
fte = spark.table("default.cleaned_fte")

In [0]:
# Filtering the datasets for ZVW declarabel no-crisis
activiteiten_zvw = activiteiten.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True)
ozp_zvw = ozp.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True).filter(F.col("crisis_binnen_budget") == False)
verblijf_zvw = verblijf.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True)

# Check if tarief_prijspeil_2026 is the same as bedrag_totaal_nza in 2026 so that we can conclude the tarief_prijspeil_2026 is correct
activiteiten_zvw.filter(F.col("jaar") == 2026).groupBy("jaar").agg(F.sum("bedrag_totaal_nza"), F.sum("tarief_prijspeil_2026")).show()

First aggregating the data

In [0]:
# Combine the ZVW declarabel no crisis datasets into one
combined_zvw = activiteiten_zvw.select("rapportagedatum", "tarief_prijspeil_2026").union(ozp_zvw.select("rapportagedatum", "tarief_prijspeil_2026")).union(verblijf_zvw.select("rapportagedatum","tarief_prijspeil_2026"))

# Group by date
daily_revenue = (combined_zvw
                 .groupBy("rapportagedatum")
                 .agg(F.round(F.sum("tarief_prijspeil_2026"), 2).alias("totale_omzet"))
                 .orderBy("rapportagedatum"))

# Seasonality analysis
Will be used for the SARIMAX baseline model

In [0]:
# ── 0. PySpark → Pandas ────────────────────────────────────────
df = daily_revenue.toPandas()
df['rapportagedatum'] = pd.to_datetime(df['rapportagedatum'])
df = df.sort_values('rapportagedatum').set_index('rapportagedatum')
df = df.asfreq('D')  # expliciete dagelijkse frequentie

# Vul eventuele missing days op (weekenden / feestdagen zonder activiteiten)
df['totale_omzet'] = df['totale_omzet'].fillna(0)

# ── 1. Ruwe tijdreeks ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['totale_omzet'], linewidth=0.8, color='steelblue')
ax.set_title('Dagelijkse omzet (2023–2025)', fontsize=13)
ax.set_ylabel('Omzet (€)')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── 2. Weeklyse aggregatie (minder ruis, beter voor seasonality) ──
weekly = df['totale_omzet'].resample('W').sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(weekly.index, weekly.values, linewidth=1.2, color='steelblue')
ax.set_title('Wekelijkse omzet (2023–2025)', fontsize=13)
ax.set_ylabel('Omzet (€)')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── 3. ACF en PACF ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Dagelijks
plot_acf(df['totale_omzet'],  lags=60, ax=axes[0, 0], title='ACF – Dagelijks (60 lags)')
plot_pacf(df['totale_omzet'], lags=60, ax=axes[0, 1], title='PACF – Dagelijks (60 lags)')

# Wekelijks
plot_acf(weekly,  lags=52, ax=axes[1, 0], title='ACF – Wekelijks (52 lags)')
plot_pacf(weekly, lags=52, ax=axes[1, 1], title='PACF – Wekelijks (52 lags)')

plt.tight_layout()
plt.show()

# ── 4. Seizoensdecompositie ────────────────────────────────────
# Wekelijks met jaarlijkse seasonaliteit (period=52)
decomp = seasonal_decompose(
    weekly,
    model='additive',
    period=52,
    extrapolate_trend='freq'
)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed',  color='steelblue')
decomp.trend.plot(   ax=axes[1], title='Trend',     color='darkorange')
decomp.seasonal.plot(ax=axes[2], title='Seasonality (period=52)', color='green')
decomp.resid.plot(   ax=axes[3], title='Residual',  color='red')
for ax in axes:
    ax.set_ylabel('Omzet (€)')
plt.suptitle('Seizoensdecompositie – Wekelijkse omzet', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── 5. Stationariteitstest (ADF) ───────────────────────────────
for label, series in [('Dagelijks', df['totale_omzet']), ('Wekelijks', weekly)]:
    adf_stat, p_value, _, _, critical_values, _ = adfuller(series.dropna())
    print(f"\n{'─'*40}")
    print(f"ADF Test – {label}")
    print(f"  ADF Statistic : {adf_stat:.4f}")
    print(f"  p-value       : {p_value:.4f}")
    print(f"  Kritieke waarden:")
    for key, val in critical_values.items():
        print(f"    {key}: {val:.4f}")
    if p_value < 0.05:
        print("  → Stationair (geen differentiatie nodig)")
    else:
        print("  → Niet stationair (overweeg d=1 in SARIMAX)")

# ── 6. Gemiddeld dagpatroon binnen de week ─────────────────────
df['weekday'] = df.index.day_name()
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_avg = df.groupby('weekday')['totale_omzet'].mean().reindex(weekday_order)

fig, ax = plt.subplots(figsize=(8, 4))
weekday_avg.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Gemiddelde omzet per weekdag', fontsize=13)
ax.set_ylabel('Gemiddelde omzet (€)')
ax.set_xlabel('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── 7. Gemiddeld maandpatroon ──────────────────────────────────
df['month'] = df.index.month
month_avg = df.groupby('month')['totale_omzet'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(8, 4))
month_avg.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Gemiddelde omzet per maand', fontsize=13)
ax.set_ylabel('Gemiddelde omzet (€)')
ax.set_xticklabels(month_labels, rotation=45)
plt.tight_layout()
plt.show()

# Baseline model: Feature engineering

In [0]:
# PySpark → Pandas
df = daily_revenue.toPandas()
df['rapportagedatum'] = pd.to_datetime(df['rapportagedatum'])
df = df.sort_values('rapportagedatum').set_index('rapportagedatum')
df = df.asfreq('D').fillna(0)

# Calendar features
df['weekday']     = df.index.dayofweek
df['month']       = df.index.month
df['week']        = df.index.isocalendar().week.astype(int)
df['year']        = df.index.year
df['quarter']     = df.index.quarter
df['day_of_year'] = df.index.dayofyear

# Cyclic encoding
df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
df['week_sin']    = np.sin(2 * np.pi * df['week'] / 52)
df['week_cos']    = np.cos(2 * np.pi * df['week'] / 52)

# Binary indicators 
df['is_weekend'] = (df['weekday'] >= 5).astype(int)
df['is_monday']  = (df['weekday'] == 0).astype(int)
df['is_friday']  = (df['weekday'] == 4).astype(int)

# Holidays
vakanties_noord = [
    # 2022
    ('voorjaar',  '2022-02-19', '2022-02-27'),
    ('mei',       '2022-04-23', '2022-05-08'),
    ('zomer',     '2022-07-09', '2022-08-21'),
    ('herfst',    '2022-10-15', '2022-10-23'),
    ('kerst',     '2022-12-24', '2023-01-08'),
    # 2023
    ('voorjaar',  '2023-02-25', '2023-03-05'),
    ('mei',       '2023-04-22', '2023-04-30'),
    ('zomer',     '2023-07-08', '2023-08-20'),
    ('herfst',    '2023-10-14', '2023-10-22'),
    ('kerst',     '2023-12-23', '2024-01-07'),
    # 2024
    ('voorjaar',  '2024-02-17', '2024-02-25'),
    ('mei',       '2024-04-27', '2024-05-05'),
    ('zomer',     '2024-07-20', '2024-09-01'),
    ('herfst',    '2024-10-19', '2024-10-27'),
    ('kerst',     '2024-12-21', '2025-01-05'),
    # 2025
    ('voorjaar',  '2025-02-22', '2025-03-02'),
    ('mei',       '2025-04-19', '2025-04-27'),
    ('zomer',     '2025-07-19', '2025-08-31'),
    ('herfst',    '2025-10-18', '2025-10-26'),
    ('kerst',     '2025-12-20', '2026-01-04'),
]

# Initialize columns on 0
for vtype in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']:
    df[f'is_{vtype}vakantie'] = 0

# Fill based on date
for vtype, start, end in vakanties_noord:
    mask = (df.index >= start) & (df.index <= end)
    df.loc[mask, f'is_{vtype}vakantie'] = 1

# Combined indicator
vakantie_cols = [f'is_{v}vakantie' for v in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']]
df['is_schoolvakantie'] = (df[vakantie_cols].sum(axis=1) > 0).astype(int)

# Separate spare days in the Netherlands
feestdagen = pd.to_datetime([
    # 2022
    '2022-01-01', '2022-04-15', '2022-04-17', '2022-04-18',
    '2022-04-27', '2022-05-05', '2022-05-26', '2022-06-05',
    '2022-06-06', '2022-12-25', '2022-12-26',
    # 2023
    '2023-01-01', '2023-04-07', '2023-04-09', '2023-04-10',
    '2023-04-27', '2023-05-05', '2023-05-18', '2023-05-28',
    '2023-05-29', '2023-12-25', '2023-12-26',
    # 2024
    '2024-01-01', '2024-03-29', '2024-03-31', '2024-04-01',
    '2024-04-27', '2024-05-05', '2024-05-09', '2024-05-19',
    '2024-05-20', '2024-12-25', '2024-12-26',
    # 2025
    '2025-01-01', '2025-04-18', '2025-04-20', '2025-04-21',
    '2025-04-26', '2025-05-05', '2025-05-29', '2025-06-08',
    '2025-06-09', '2025-12-25', '2025-12-26',
])
df['is_feestdag'] = df.index.isin(feestdagen).astype(int)

# Day after spare day
df['is_dag_na_feestdag'] = df['is_feestdag'].shift(1).fillna(0).astype(int)

# Contractual consumption
max_annual_revenue = df.groupby('year')['totale_omzet'].sum().max()
df['cumulative_revenue_ytd'] = df.groupby('year')['totale_omzet'].cumsum()
df['contractual_consumption_rate'] = (
    df['cumulative_revenue_ytd'] / max_annual_revenue
).clip(0, 1)

# Trend feature
df['time_index'] = (df.index - df.index.min()).days

# Drop null values
df_features = df.dropna()

# Check
print(f"Shape: {df_features.shape}")
print(f"\nFeatures ({len(df_features.columns)}):")
print(df_features.columns.tolist())
print(f"\nPeriode: {df_features.index.min().date()} → {df_features.index.max().date()}")
print(f"\nMissing values:\n{df_features.isnull().sum()[df_features.isnull().sum() > 0]}")

# Sanity check: holidays per type
print("\nVakantiedagen per type:")
for col in vakantie_cols + ['is_schoolvakantie', 'is_feestdag']:
    print(f"  {col}: {df_features[col].sum()} dagen")

# Baseline model: finding the best SARIMAX model

In [0]:
# ── 0. Split ───────────────────────────────────────────────────
train = df_features[df_features['year'].isin([2023, 2024])]
test  = df_features[df_features['year'] == 2025]

print(f"Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} dagen)")
print(f"Test:  {test.index.min().date()} → {test.index.max().date()} ({len(test)} dagen)")

# ── 1. Exogene variabelen ──────────────────────────────────────
exog_cols = [
    'is_feestdag',
    'is_dag_na_feestdag',
    'is_schoolvakantie',
]

# Train test split
y_train = train['totale_omzet']
y_test  = test['totale_omzet']
X_train = train[exog_cols]
X_test  = test[exog_cols]

# Rescale because Sarimax does not work with big values
scale_factor = 1000
y_train_scaled = y_train / scale_factor
y_test_scaled  = y_test  / scale_factor

# ── 2. AIC Grid Search via expanding window CV ────────────────
print("\nAIC Grid Search gestart (dit duurt ~5-10 minuten)...")

# Expanding window CV op trainset
tscv = TimeSeriesSplit(n_splits=2, gap=0)
results_grid = []

results_grid = []

# Extra 'd' loop toegevoegd voor trend-differencing
for p in [0, 1]:
    for d in [0, 1]: # <--- Nieuw: test nu ook non-stationaire trends!
        for q in [0, 1]:
            for P in [0, 1]:
                for D in [0, 1]:
                    for Q in [0, 1]:
                        fold_maes = []
                        try:
                            for fold, (tr_idx, val_idx) in enumerate(tscv.split(y_train_scaled)):
                                y_tr  = y_train_scaled.iloc[tr_idx]
                                y_val = y_train_scaled.iloc[val_idx]
                                X_tr  = X_train.iloc[tr_idx]
                                X_val = X_train.iloc[val_idx]

                                model = SARIMAX(
                                    y_tr,
                                    exog=X_tr,
                                    order=(p, d, q), # <--- Nu met variabele 'd'
                                    seasonal_order=(P, D, Q, 7),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False
                                )
                                res = model.fit(disp=False, maxiter=200)
                                pred = res.forecast(steps=len(y_val), exog=X_val)
                                pred = pred.clip(lower=0)

                                mae = mean_absolute_error(y_val, pred)
                                fold_maes.append(mae)

                            results_grid.append({
                                'order': (p, d, q),
                                'seasonal': (P, D, Q, 7),
                                'mean_cv_MAE': round(np.mean(fold_maes) * scale_factor, 0),
                                'std_cv_MAE':  round(np.std(fold_maes)  * scale_factor, 0)
                            })
                            print(f"  order={(p,d,q)} seasonal={(P,D,Q,7)} → CV MAE: €{np.mean(fold_maes)*scale_factor:,.0f}")
                        except Exception as e:
                            # Print bewust uitgezet om de logs schoon te houden bij foutieve wiskundige combinaties
                            continue

# Beste model
results_df = pd.DataFrame(results_grid).sort_values('mean_cv_MAE')
print("\nTop 5:")
print(results_df.head(5).to_string(index=False))

best_order    = results_df.iloc[0]['order']
best_seasonal = results_df.iloc[0]['seasonal']

print(f"\nBeste specificatie: order={best_order}, seasonal={best_seasonal}")

# ── 3. Finaal model trainen op volledige trainset ──────────────
print("\nFinaal model trainen...")

final_model = SARIMAX(
    y_train_scaled,
    exog=X_train,
    order=best_order,
    seasonal_order=best_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_result = final_model.fit(disp=False)
print(final_result.summary())

# ── 4. Test voorspelling (2025) ────────────────────────────────
test_pred_scaled = final_result.forecast(
    steps=len(test),
    exog=X_test
)

# <--- CRUCIAAL: Vermenigvuldig de voorspelling weer met 1000 om in Euro's te komen
test_pred = test_pred_scaled * scale_factor 
test_pred = pd.Series(test_pred.values, index=y_test.index)
test_pred = test_pred.clip(lower=0)

In [0]:
# Safe results 
import os

# Create submap
pad_sarimax = './sarimax'
os.makedirs(pad_sarimax, exist_ok=True)

test_evaluatie = test.copy()
test_evaluatie['Voorspelde_Omzet'] = test_pred.values

# Save dataframes
train.to_csv(f'{pad_sarimax}/train.csv', index=True)
test_evaluatie.to_csv(f'{pad_sarimax}/test.csv', index=True)
results_df.to_csv(f'{pad_sarimax}/results_df.csv', index=False)

X_train.to_csv(f'{pad_sarimax}/X_train.csv', index=True)
X_test.to_csv(f'{pad_sarimax}/X_test.csv', index=True)

# Save model
final_result.save(f'{pad_sarimax}/getraind_sarimax_model.pkl')

print(f"\n✅ Model en alle bijbehorende tabellen zijn succesvol opgeslagen in de map: {pad_sarimax}")

# Baseline model: trying if baseline is better when using separate SARIMAX models per insurer and aggregate them 

In [0]:
# Agg data with insurer
combined_zvw_insurer = activiteiten_zvw.select("rapportagedatum", "tarief_prijspeil_2026", "financier").union(ozp_zvw.select("rapportagedatum", "tarief_prijspeil_2026", "financier")).union(verblijf_zvw.select("rapportagedatum","tarief_prijspeil_2026", "financier"))

overig_lijst = [
    'ASIELZOEKERS', 
    'CAK', 
    'OVERIG', 
    'BUITENLAND', 
    'KRIJGSMACHT', 
    'Onbekend'
]

combined_zvw_insurer = combined_zvw_insurer.withColumn(
    'financier',
    F.when(F.col('financier').isin(overig_lijst), 'OVERIG')
     .otherwise(F.col('financier'))
)

combined_zvw_insurer = combined_zvw_insurer.groupby("rapportagedatum", "financier").agg(F.sum("tarief_prijspeil_2026").alias("totale_omzet"))

display(combined_zvw_insurer.limit(10))

Define functions for later use

In [0]:
# Define function
def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    print(f"\n{'─'*45}")
    print(f"Metrics – {label}")
    print(f"  MAE   : €{mae:,.0f}")
    print(f"  RMSE  : €{rmse:,.0f}")
    print(f"  MAPE  : {mape:.1f}%")
    print(f"  sMAPE : {smape:.1f}%")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'sMAPE': smape}

def engineer_features(df_raw):
    df = df_raw.copy()
    if 'rapportagedatum' in df.columns:
        df['rapportagedatum'] = pd.to_datetime(df['rapportagedatum'])
        # Groepeer op datum om dubbele boekingen op te tellen
        df = df.groupby('rapportagedatum')['totale_omzet'].sum().reset_index()
        df = df.sort_values('rapportagedatum').set_index('rapportagedatum')
    
    # 🎯 DE FIX: Forceer de index om een officieel Datetime object te zijn
    df.index = pd.to_datetime(df.index)
    
    df = df.asfreq('D').fillna(0)
    
    # Lags & Kalender
    df['lag_1d']  = df['totale_omzet'].shift(1)
    df['lag_7d']  = df['totale_omzet'].shift(7)
    df['lag_14d'] = df['totale_omzet'].shift(14)
    df['lag_28d'] = df['totale_omzet'].shift(28)
    df['rolling_mean_7d']  = df['totale_omzet'].shift(1).rolling(7).mean()
    df['rolling_mean_28d'] = df['totale_omzet'].shift(1).rolling(28).mean()
    
    df['weekday']     = df.index.dayofweek
    df['month']       = df.index.month
    df['week']        = df.index.isocalendar().week.astype(int)
    df['year']        = df.index.year
    df['quarter']     = df.index.quarter
    df['day_of_year'] = df.index.dayofyear

    # Cyclisch & Binair
    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    df['week_sin']    = np.sin(2 * np.pi * df['week'] / 52)
    df['week_cos']    = np.cos(2 * np.pi * df['week'] / 52)
    df['is_weekend']  = (df['weekday'] >= 5).astype(int)
    df['is_monday']   = (df['weekday'] == 0).astype(int)
    df['is_friday']   = (df['weekday'] == 4).astype(int)

    # Vakanties & Feestdagen
    vakanties_noord = [
        ('voorjaar', '2022-02-19', '2022-02-27'), ('mei', '2022-04-23', '2022-05-08'), ('zomer', '2022-07-09', '2022-08-21'), ('herfst', '2022-10-15', '2022-10-23'), ('kerst', '2022-12-24', '2023-01-08'),
        ('voorjaar', '2023-02-25', '2023-03-05'), ('mei', '2023-04-22', '2023-04-30'), ('zomer', '2023-07-08', '2023-08-20'), ('herfst', '2023-10-14', '2023-10-22'), ('kerst', '2023-12-23', '2024-01-07'),
        ('voorjaar', '2024-02-17', '2024-02-25'), ('mei', '2024-04-27', '2024-05-05'), ('zomer', '2024-07-20', '2024-09-01'), ('herfst', '2024-10-19', '2024-10-27'), ('kerst', '2024-12-21', '2025-01-05'),
        ('voorjaar', '2025-02-22', '2025-03-02'), ('mei', '2025-04-19', '2025-04-27'), ('zomer', '2025-07-19', '2025-08-31'), ('herfst', '2025-10-18', '2025-10-26'), ('kerst', '2025-12-20', '2026-01-04'),
    ]
    for vtype in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']:
        df[f'is_{vtype}vakantie'] = 0
    for vtype, start, end in vakanties_noord:
        mask = (df.index >= start) & (df.index <= end)
        df.loc[mask, f'is_{vtype}vakantie'] = 1
    vakantie_cols = [f'is_{v}vakantie' for v in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']]
    df['is_schoolvakantie'] = (df[vakantie_cols].sum(axis=1) > 0).astype(int)

    feestdagen = pd.to_datetime([
        '2022-01-01', '2022-04-15', '2022-04-17', '2022-04-18', '2022-04-27', '2022-05-05', '2022-05-26', '2022-06-05', '2022-06-06', '2022-12-25', '2022-12-26',
        '2023-01-01', '2023-04-07', '2023-04-09', '2023-04-10', '2023-04-27', '2023-05-05', '2023-05-18', '2023-05-28', '2023-05-29', '2023-12-25', '2023-12-26',
        '2024-01-01', '2024-03-29', '2024-03-31', '2024-04-01', '2024-04-27', '2024-05-05', '2024-05-09', '2024-05-19', '2024-05-20', '2024-12-25', '2024-12-26',
        '2025-01-01', '2025-04-18', '2025-04-20', '2025-04-21', '2025-04-26', '2025-05-05', '2025-05-29', '2025-06-08', '2025-06-09', '2025-12-25', '2025-12-26',
    ])
    df['is_feestdag'] = df.index.isin(feestdagen).astype(int)
    df['is_dag_na_feestdag'] = df['is_feestdag'].shift(1).fillna(0).astype(int)

    df['time_index'] = (df.index - df.index.min()).days

    return df.dropna()

def train_evaluate_sarimax(df_features, verzekeraar_naam):
    print(f"\n{'='*60}")
    print(f"🚀 START SARIMAX: {verzekeraar_naam}")
    print(f"{'='*60}")

    train = df_features[df_features['year'].isin([2023, 2024])]
    test  = df_features[df_features['year'] == 2025]

    exog_cols = ['is_feestdag', 'is_dag_na_feestdag', 'is_schoolvakantie']
    
    y_train = train['totale_omzet']
    y_test  = test['totale_omzet']
    X_train = train[exog_cols]
    X_test  = test[exog_cols]

    scale_factor = 1000
    y_train_scaled = y_train / scale_factor
    y_test_scaled  = y_test  / scale_factor

    tscv = TimeSeriesSplit(n_splits=2, gap=0)
    results_grid = []

    for p in [0, 1]:
        for d in [0, 1]: 
            for q in [0, 1]:
                for P in [0, 1]:
                    for D in [0, 1]:
                        for Q in [0, 1]:
                            fold_maes = []
                            try:
                                for fold, (tr_idx, val_idx) in enumerate(tscv.split(y_train_scaled)):
                                    model = SARIMAX(
                                        y_train_scaled.iloc[tr_idx],
                                        exog=X_train.iloc[tr_idx],
                                        order=(p, d, q),
                                        seasonal_order=(P, D, Q, 7),
                                        enforce_stationarity=False,
                                        enforce_invertibility=False
                                    )
                                    res = model.fit(disp=False, maxiter=200)
                                    pred = res.forecast(steps=len(val_idx), exog=X_train.iloc[val_idx]).clip(lower=0)
                                    fold_maes.append(mean_absolute_error(y_train_scaled.iloc[val_idx], pred))

                                results_grid.append({
                                    'order': (p, d, q),
                                    'seasonal': (P, D, Q, 7),
                                    'mean_cv_MAE': np.mean(fold_maes) * scale_factor
                                })
                            except Exception:
                                continue

    # Valback voor als de wiskunde helemaal breekt op kleine verzekeraars
    if len(results_grid) == 0:
        print("WAARSCHUWING: Grid search mislukt. Valback naar (0,0,0)(0,0,0,7)")
        best_order, best_seasonal = (0,0,0), (0,0,0,7)
    else:
        results_df = pd.DataFrame(results_grid).sort_values('mean_cv_MAE')
        best_order    = results_df.iloc[0]['order']
        best_seasonal = results_df.iloc[0]['seasonal']

    print(f"Beste specificatie: order={best_order}, seasonal={best_seasonal}")

    final_model = SARIMAX(
        y_train_scaled,
        exog=X_train,
        order=best_order,
        seasonal_order=best_seasonal,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    final_result = final_model.fit(disp=False)

    test_pred = (final_result.forecast(steps=len(test), exog=X_test) * scale_factor).clip(lower=0)
    test_pred = pd.Series(test_pred.values, index=y_test.index)

    test_metrics = evaluate(y_test, test_pred, f'Test 2025 – {verzekeraar_naam}')
    
    werk = y_test.sum()
    voor = test_pred.sum()
    print(f"  Jaaromzet Werkelijk : €{werk:,.0f}")
    print(f"  Jaaromzet Voorspeld : €{voor:,.0f}")

    return {
        'verzekeraar': verzekeraar_naam,
        'metrics': test_metrics,
        'pred': test_pred,
        'actual': y_test,
        'jaaromzet_werkelijk': werk,
        'jaaromzet_voorspeld': voor
    }


In [0]:
# ── 2. PySpark naar Pandas & Dataset Preparatie ──────────────────────────────
print("Data ophalen uit PySpark...")
df_compleet = combined_zvw_insurer.toPandas()

datasets_dict = {}
verzekeraars = df_compleet['financier'].unique()

print("Feature Engineering toepassen per financier...")
for vz in verzekeraars:
    # Filter per financier op de pas hernoemde kolommen
    df_vz_ruw = df_compleet[df_compleet['financier'] == vz][['rapportagedatum', 'totale_omzet']]
    datasets_dict[vz] = engineer_features(df_vz_ruw)

print(f"Klaar! {len(datasets_dict)} datasets geprepareerd.")

print("\nOverzicht van de opgeslagen datasets:")
for naam, df in datasets_dict.items():
    print(f"Verzekeraar: {naam:<15} | Shape: {df.shape[0]} rijen, {df.shape[1]} kolommen")

In [0]:
# ── 3. Train Modellen (De Loop) ──────────────────────────────────────────────
alle_resultaten = {}

for naam, df_inz in datasets_dict.items():
    alle_resultaten[naam] = train_evaluate_sarimax(df_inz, naam)

# ── 4. Aggregatie & Finale Rapportage (Bottom-up) ────────────────────────────
print("\n\n" + "█"*60)
print("RAPPORTAGE PER VERZEKERAAR (JAAROMZET 2025)")
print("█"*60)

eerste_key = list(alle_resultaten.keys())[0]
totale_dag_voorspelling = pd.Series(0.0, index=alle_resultaten[eerste_key]['pred'].index)
totale_dag_werkelijkheid = pd.Series(0.0, index=alle_resultaten[eerste_key]['actual'].index)

for naam, res in alle_resultaten.items():
    werk = res['jaaromzet_werkelijk']
    voor = res['jaaromzet_voorspeld']
    verschil = voor - werk
    perc_verschil = (verschil / werk) * 100 if werk > 0 else 0
    
    # Zorg dat alle series dezelfde index hebben voordat we ze optellen
    totale_dag_voorspelling = totale_dag_voorspelling.add(res['pred'], fill_value=0)
    totale_dag_werkelijkheid = totale_dag_werkelijkheid.add(res['actual'], fill_value=0)

    print(f"{naam:<18} | Werkelijk: €{werk:>11,.0f} | Voorspeld: €{voor:>11,.0f} | Verschil: {perc_verschil:>+6.1f}%")

print("\n\n" + "█"*60)
print("GEAGGREGEERDE TOTALE OMZET (ALLE VERZEKERAARS GECOMBINEERD)")
print("█"*60)

globale_werkelijke_jaaromzet = totale_dag_werkelijkheid.sum()
globale_voorspelde_jaaromzet = totale_dag_voorspelling.sum()
globaal_verschil = globale_voorspelde_jaaromzet - globale_werkelijke_jaaromzet
globaal_perc = (globaal_verschil / globale_werkelijke_jaaromzet) * 100 if globale_werkelijke_jaaromzet > 0 else 0

print(f"Totale Jaaromzet Werkelijk : €{globale_werkelijke_jaaromzet:,.0f}")
print(f"Totale Jaaromzet Voorspeld : €{globale_voorspelde_jaaromzet:,.0f}")
print(f"Verschil                   : €{globaal_verschil:,.0f} ({globaal_perc:+.1f}%)")

finale_baseline_metrics = evaluate(
    totale_dag_werkelijkheid, 
    totale_dag_voorspelling, 
    'Bottom-up SARIMAX Totaal 2025'
)

it's clear that for smallers insurers, this method does not work, porbably because the client population varies a lot during the specific periods. Therefore, a machine learning forecast method could include the kind of client population and improve the forecast.

This split method also performs worse than the Top-down method.
Takes a lot of time

# Statistical analysis for feature selection

In [0]:
# Select columns for ML approach
ML_columns = [
    'beroepscategorie',
    'org_level_1',
    'setting',
    'consult_type',
    'financier',
    'diagnosegroep',
    'tijd_client_direct_minuten',
    'tarief_prijspeil_2026'
]

In [0]:
def analyseer_omzet_drivers(df_spark, dataset_naam, cat_cols, target_col='tarief_prijspeil_2026', sample_size=100000):
    """
    Functie om statistisch te testen welke categorische kolommen de meeste 
    invloed hebben op de omzet (target_col) in een PySpark DataFrame.
    """
    print(f"\n" + "="*70)
    print(f"START ANALYSE: {dataset_naam.upper()}")
    print("="*70)
    
    # ── 1. Steekproef ophalen ──────────────────────────────────────────────
    geschat_aantal_rijen = df_spark.count()
    fractie = sample_size / geschat_aantal_rijen if geschat_aantal_rijen > sample_size else 1.0
    
    print(f"Data ophalen (fractie: {fractie:.4f}, max {sample_size} rijen)...")
    df_sample = df_spark.sample(withReplacement=False, fraction=fractie, seed=42).toPandas()
    
    # ── 2. Bewijs van de scheve verdeling ──────────────────────────────────
    print("Verdeling plotten...")
    plt.figure(figsize=(10, 6))
    max_val = df_sample[target_col].quantile(0.99) 
    plot_data = df_sample[df_sample[target_col] <= max_val][target_col]
    
    sns.histplot(plot_data, bins=100, kde=True, color='royalblue')
    plt.title(f'Verdeling van Omzet per Transactie - {dataset_naam} (Rechtsscheef)')
    plt.xlabel('Omzet / Tarief (€)')
    plt.ylabel('Aantal transacties')
    plt.show()
    
    # ── 3. Kruskal-Wallis Test & Effectgrootte ─────────────────────────────
    print("Kruskal-Wallis statistische tests uitvoeren...")
    results = []
    n = len(df_sample)
    
    for col in cat_cols:
        # Veiligheidscheck: bestaat de kolom wel in deze dataset?
        if col not in df_sample.columns:
            print(f"  [!] Let op: Kolom '{col}' overgeslagen (niet gevonden in {dataset_naam}).")
            continue
            
        df_clean = df_sample.dropna(subset=[col, target_col])
        groups = [group[target_col].values for name, group in df_clean.groupby(col) if len(group) > 5]
        
        if len(groups) >= 2:
            stat, p_val = stats.kruskal(*groups)
            epsilon_sq = stat / (n - 1)
            
            results.append({
                'Variabele': col,
                'H-Statistiek': round(stat, 2),
                'P-Waarde': p_val,
                'Effectgrootte (Epsilon²)': round(epsilon_sq, 4)
            })
            
    # ── 4. Resultaten presenteren ──────────────────────────────────────────
    if not results:
        print("Geen resultaten om te tonen (geen geldige kolommen of te weinig data).")
        return None
        
    df_results = pd.DataFrame(results).sort_values(by='Effectgrootte (Epsilon²)', ascending=False)
    
    print("\n" + "█"*60)
    print(f"STATISTISCHE INVLOED OP OMZET - {dataset_naam.upper()}")
    print("█"*60)
    display(df_results)
    
    return df_results

In [0]:
activiteiten_zvw.columns
cols_activiteiten = ['beroepscategorie', 'org_level_1', 'setting', 'consult_type', 'diagnosegroep', 'diagnose', 'beroepcode', 'org_level_3', 'groepsgrootte','medewerkerberoep', 'no_show']
analyseer_omzet_drivers(activiteiten_zvw, "Activiteiten", cols_activiteiten)

In [0]:
cols_verblijf = ['org_level_1', 'org_level_3', 'zorgvraagtypering', 'beveiligde_setting', 'cruciale_zorg', 'meer_dan_365_dagen_zvw_verblijf']
analyseer_omzet_drivers(verblijf_zvw, "Verblijf", cols_verblijf)

In [0]:
cols_ozp = ['contactomschrijving', 'beroepcode', 'beroepscategorie', 'org_level_1', 'org_level_3', 'no_show', 'meer_dan_365_dagen_verblijf', 'zorgvraagtypering', 'declaratiecode']
analyseer_omzet_drivers(ozp_zvw, "OZP", cols_ozp)

In [0]:
def categorische_correlatie_matrix(df_spark, cat_cols, dataset_naam="Dataset", sample_size=100000):
    """
    Haalt een steekproef uit een PySpark DataFrame, berekent Cramér's V 
    tussen alle opgegeven kolommen en plot een heatmap.
    """
    print("\n" + "="*70)
    print(f"CATEGORISCHE CORRELATIEMATRIX (CRAMÉR'S V) - {dataset_naam.upper()}")
    print("="*70)
    
    # ── 1. Steekproef ophalen (direct vanuit PySpark) ──────────────────────
    geschat_aantal_rijen = df_spark.count()
    fractie = sample_size / geschat_aantal_rijen if geschat_aantal_rijen > sample_size else 1.0
    
    print(f"Data ophalen (fractie: {fractie:.4f}, max {sample_size} rijen)...")
    df_sample = df_spark.sample(withReplacement=False, fraction=fractie, seed=42).toPandas()
    
    # ── 2. Matrix voorbereiden ─────────────────────────────────────────────
    # Check welke kolommen daadwerkelijk in de dataset zitten
    cols_to_use = [col for col in cat_cols if col in df_sample.columns]
    
    if len(cols_to_use) < 2:
        print("[!] Niet genoeg geldige kolommen gevonden om een matrix te maken.")
        return None
        
    corr_matrix = pd.DataFrame(index=cols_to_use, columns=cols_to_use, dtype=float)
    
    # Interne functie om Cramér's V te berekenen (met bias-correctie)
    def cramers_v(x, y):
        confusion_matrix = pd.crosstab(x, y)
        chi2 = ss.chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        phi2 = chi2 / n
        r, k = confusion_matrix.shape
        if r <= 1 or k <= 1:
            return 0.0
        phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
        rcorr = r - ((r-1)**2)/(n-1)
        kcorr = k - ((k-1)**2)/(n-1)
        return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

    # ── 3. Matrix vullen ───────────────────────────────────────────────────
    print("Correlaties berekenen...")
    for col1 in cols_to_use:
        for col2 in cols_to_use:
            if col1 == col2:
                corr_matrix.loc[col1, col2] = 1.0
            else:
                df_clean = df_sample.dropna(subset=[col1, col2])
                corr_matrix.loc[col1, col2] = cramers_v(df_clean[col1], df_clean[col2])
                
    # ── 4. Plot de Heatmap ─────────────────────────────────────────────────
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".2f", linewidths=.5)
    plt.title(f"Onderlinge Correlatie Categorieën - {dataset_naam}")
    plt.tight_layout()
    plt.show()
    
    return corr_matrix

In [0]:
# Choose the columns with > 0,1
cols_activiteiten = ['consult_type', 'groepsgrootte', 'setting', 'org_level_3', 'beroepcode', 'diagnose']
categorische_correlatie_matrix(activiteiten_zvw, cols_activiteiten, "Activiteiten")

In [0]:
# Choose columns with > 0,1 for Verblijf results in only org_level_3
# Choose columns with > 0,1 for ozp
cols_ozp = ['org_level_3', 'org_level_1', 'beroepcode', 'beroepscategorie', 'declaratiecode']
categorische_correlatie_matrix(ozp_zvw, cols_ozp, "OZP")


In [0]:
# filter hr data
df_hr_week = productiviteit.filter(F.col("functie_groep") == "Behandeling") \
    .withColumn("week_start", F.date_trunc("week", F.col("rapportagedatum"))) \
    .groupBy("week_start").agg(
        F.sum("uren_bruto").alias("uren_bruto"),
        F.sum("uren_netto").alias("uren_netto")
    )

# filter directe uren en omzet
df_act_week = activiteiten_zvw.withColumn(
    "week_start", F.date_trunc("week", F.col("rapportagedatum"))
).groupBy("week_start").agg(
    F.sum("tijd_client_direct_minuten").alias("totale_minuten"),
    F.sum("tarief_prijspeil_2026").alias("totale_omzet")
)

# join 
df_week_spark = df_act_week.join(df_hr_week, on="week_start", how="inner")
df_week_spark_clean = df_week_spark.filter(F.col("uren_netto") > 5000)

# to pandas
df_week = df_week_spark_clean.dropna().orderBy("week_start").toPandas()

kolommen_voor_correlatie = ["uren_bruto", "uren_netto", "totale_minuten", "totale_omzet"]
for col in kolommen_voor_correlatie:
    df_week[col] = pd.to_numeric(df_week[col], errors='coerce')
df_week = df_week.dropna(subset=kolommen_voor_correlatie)
corr_matrix = df_week[kolommen_voor_correlatie].corr(method='pearson')

# Visualisatie 1: Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Pearson Correlatie: HR (Behandeling) vs Productie per Week')
plt.tight_layout()
plt.show()

# Visualisatie 2: Regressielijn
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_week, 
    x='uren_netto', 
    y='totale_minuten', 
    scatter_kws={'alpha':0.7, 'color':'#16a085', 's': 60}, 
    line_kws={'color':'#e74c3c', 'linewidth':2}
)
plt.title('Regressie-analyse: Wekelijkse Netto Uren (Behandelaren) vs Zorgminuten')
plt.xlabel('Week Totaal Netto Uren (Alleen Behandelaren)')
plt.ylabel('Week Totaal Directe Minuten (Productie)')
plt.grid(True, linestyle='--', alpha=0.6)
df_week['uren_netto'] = pd.to_numeric(df_week['uren_netto'], errors='coerce')
df_week['totale_minuten'] = pd.to_numeric(df_week['totale_minuten'], errors='coerce')
df_week = df_week.dropna(subset=['uren_netto', 'totale_minuten'])
plt.show()

In [0]:
display(diensten.limit(10))

In [0]:
display(diensten.select("dienst_hoofdsoort").filter(F.col("werkzame_uren") == "true").distinct())

In [0]:
hr_stamgegevens = productiviteit.select(
    "personeelsnummer",
    "rapportagedatum",
    "functie",
    "functie_groep", 
    "beroepscategorie",
    "org_level_3",
)

hr_stamgegevens_uniek = hr_stamgegevens.dropDuplicates(["personeelsnummer", "rapportagedatum", "org_level_3"])

diensten_verrijkt = diensten.join(
    hr_stamgegevens_uniek,
    on=["personeelsnummer", "rapportagedatum", "org_level_3"],
    how="left"
)

diensten_behandeling = diensten_verrijkt.filter(F.col("functie_groep") == "Behandeling")

In [0]:
display(diensten_behandeling.select("dienst_hoofdsoort").filter(F.col("werkzame_uren") == "true").distinct())

In [0]:
print("█"*70)
print("CAPACITEIT VS PRODUCTIE: DE MOTOR VAN DE ORGANISATIE")
print("█"*70)

datum_kolom_act = "rapportagedatum"  
datum_kolom_hr = "rapportagedatum"   

# ── 2. Filter en Aggregeer HR-data (Behandelaren) naar Weekniveau ──────────
print("1. HR-data filteren op 'Behandeling' en groeperen per week...")
geldige_diensten = [
    "Meeruren BETAALD",
    "Overwerk BETAALD",
    "Werk t Crisis-TG",
    "Meeruren TIJD",
    "Gewerkte uren",
    "Werk t AW-TG",
    "Werk t AW-TT",
    "Werk t Crisis-TT",
    "Werk t Crisis-GG",
    "Overwerk TIJD",
    "Werk t AW-GG"
]

df_hr_week = diensten_behandeling.filter(F.col("werkzame_uren") == "true") \
    .filter(F.col("dienst_hoofdsoort").isin(geldige_diensten)) \
    .withColumn("week_start", F.date_trunc("week", F.col(datum_kolom_hr))) \
    .groupBy("week_start").agg(
        F.sum("dienst_uren").alias("dienst_uren")
    )
# ── 3. Aggregeer Activiteiten naar Weekniveau ──────────────────────────────
print("2. Activiteiten groeperen per week...")
df_act_week = activiteiten_zvw.withColumn(
    "week_start", F.date_trunc("week", F.col(datum_kolom_act))
).groupBy("week_start").agg(
    F.sum("tijd_client_direct_minuten").alias("totale_minuten"),
    F.sum("tarief_prijspeil_2026").alias("totale_omzet")
)

# ── 4. Datasets samenvoegen en opschonen ───────────────────────────────────
print("3. Datasets samenvoegen...")
df_week_spark = df_act_week.join(df_hr_week, on="week_start", how="inner")

# Naar Pandas en sorteren op datum
df_week = df_week_spark.dropna().orderBy("week_start").toPandas()
print(f"--> Succes! {len(df_week)} schone weken overgehouden voor analyse.\n")

# ── 5. Pearson Correlatie & Visualisaties ──────────────────────────────────
if not df_week.empty:
    kolommen_voor_correlatie = ["dienst_uren", "totale_minuten", "totale_omzet"]
    for col in kolommen_voor_correlatie:
        df_week[col] = pd.to_numeric(df_week[col], errors='coerce')
    df_week = df_week.dropna(subset=kolommen_voor_correlatie)
    corr_matrix = df_week[kolommen_voor_correlatie].corr(method='pearson')

    # Visualisatie 1: Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
    plt.title('Pearson Correlatie: HR (Behandeling) vs Productie per Week')
    plt.tight_layout()
    plt.show()

    # Visualisatie 2: Regressielijn
    plt.figure(figsize=(10, 6))
    sns.regplot(
        data=df_week, 
        x='dienst_uren', 
        y='totale_minuten', 
        scatter_kws={'alpha':0.7, 'color':'#16a085', 's': 60}, 
        line_kws={'color':'#e74c3c', 'linewidth':2}
    )
    plt.title('Regressie-analyse: Wekelijkse Netto Uren (Behandelaren) vs Zorgminuten')
    plt.xlabel('Week Totaal Netto Uren (Alleen Behandelaren)')
    plt.ylabel('Week Totaal Directe Minuten (Productie)')
    plt.grid(True, linestyle='--', alpha=0.6)
    df_week['dienst_uren'] = pd.to_numeric(df_week['dienst_uren'], errors='coerce')
    df_week['totale_minuten'] = pd.to_numeric(df_week['totale_minuten'], errors='coerce')
df_week = df_week.dropna(subset=['dienst_uren', 'totale_minuten'])
plt.show()

In [0]:
fte_behandeling = fte.filter(F.col("functie_groep") == "Behandeling")

df_fte_week = fte_behandeling.withColumn("week_start", F.date_trunc("week", F.col("rapportagedatum"))) \
    .groupBy("week_start").agg(
        F.sum("fte_bruto_contract").alias("fte_bruto_contract"),
        F.sum("fte_netto_contract").alias("fte_netto_contract"),
        F.sum("fte_inzet").alias("fte_inzet")
    )

# ── 3. Aggregeer Activiteiten naar Weekniveau ──────────────────────────────
print("2. Activiteiten groeperen per week...")
df_act_week = activiteiten_zvw.withColumn(
    "week_start", F.date_trunc("week", F.col(datum_kolom_act))
).groupBy("week_start").agg(
    F.sum("tijd_client_direct_minuten").alias("totale_minuten"),
    F.sum("tarief_prijspeil_2026").alias("totale_omzet")
)

# ── 4. Datasets samenvoegen en opschonen ───────────────────────────────────
print("3. Datasets samenvoegen...")
df_week_spark = df_act_week.join(df_fte_week, on="week_start", how="inner")
df_week_spark_clean = df_week_spark.filter(F.col("fte_inzet") > 1000)

# Naar Pandas en sorteren op datum
df_week = df_week_spark_clean.dropna().orderBy("week_start").toPandas()
print(f"--> Succes! {len(df_week)} schone weken overgehouden voor analyse.\n")

# ── 5. Pearson Correlatie & Visualisaties ──────────────────────────────────

kolommen_voor_correlatie = ["fte_bruto_contract", "fte_netto_contract", "fte_inzet", "totale_minuten", "totale_omzet"]
for col in kolommen_voor_correlatie:
    df_week[col] = pd.to_numeric(df_week[col], errors='coerce')
df_week = df_week.dropna(subset=kolommen_voor_correlatie)
corr_matrix = df_week[kolommen_voor_correlatie].corr(method='pearson')

# Visualisatie 1: Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Pearson Correlatie: HR (Behandeling) vs Productie per Week')
plt.tight_layout()
plt.show()

# Visualisatie 2: Regressielijn
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_week, 
    x='fte_inzet', 
    y='totale_minuten', 
    scatter_kws={'alpha':0.7, 'color':'#16a085', 's': 60}, 
    line_kws={'color':'#e74c3c', 'linewidth':2}
)
plt.title('Regressie-analyse: Wekelijkse Netto Uren (Behandelaren) vs Zorgminuten')
plt.xlabel('Week Totaal Netto Uren (Alleen Behandelaren)')
plt.ylabel('Week Totaal Directe Minuten (Productie)')
plt.grid(True, linestyle='--', alpha=0.6)
df_week['fte_inzet'] = pd.to_numeric(df_week['fte_inzet'], errors='coerce')
df_week['totale_minuten'] = pd.to_numeric(df_week['totale_minuten'], errors='coerce')
df_week = df_week.dropna(subset=['fte_inzet', 'totale_minuten'])
plt.show()

In [0]:
df_week

# Model architecture
I will produce one dataset for the XGBoost model, because they XGBoost could easily learn the different forecasting methods per declaratiecode_categorie
### Generate and combine the dataset

In [0]:
# Select columns which can be used for the model development and output columns time and revenue
df_act = activiteiten_zvw.select(
    F.col("rapportagedatum"),
    F.col("beroepscategorie"),
    F.col("medewerkerberoep"),
    F.col("org_level_1"),
    F.col("org_level_2"),
    F.col("org_level_3"),
    F.col("tijd_client_direct_minuten"),
    F.col("declaratiecode_categorie"),
    F.col("groepsgrootte"),
    F.col("setting"),
    F.col("consult_type"),
    F.col("financier"),
    F.col("diagnose"),
    F.col("diagnosegroep"),
    F.col("meer_dan_365_dagen_zvw_verblijf"),
    F.col("tarief_prijspeil_2026"),
    F.col("client_code")
)

df_verblijf = verblijf_zvw.select(
    F.col("rapportagedatum"),
    F.col("declaratiecode_categorie"),
    F.col("financier"),
    F.col("org_level_1"),
    F.col("org_level_2"),
    F.col("org_level_3"),
    F.col("meer_dan_365_dagen_zvw_verblijf"),
    F.col("tarief_prijspeil_2026"),
    F.col("client_code")
)

df_verblijf = df_verblijf.withColumn(
    "available_beds",
    F.when(F.col("org_level_3") == "12102 - Kliniek ouderen Amsterdam", F.lit(21))
    .when(F.col("org_level_3") == "11404 - Vervolgkliniek 2 Bennebroek", F.lit(20))
    .when(F.col("org_level_3") == "8520 - Kliniek gesloten ouderen Amsterdam", F.lit(21))
    .when(F.col("org_level_3") == "7550 - Bocholtstraat Amsterdam", F.lit(0))               # set to 0 because Bocholtstraat only has WLZ beds
    .when(F.col("org_level_3") == "7252 - Kliniek gesloten Amsterdam", F.lit(21))
    .when(F.col("org_level_3") == "11305 - Kliniek Gesloten Haarlem", F.lit(24))
    .when(F.col("org_level_3") == "12303 - Kliniek Amstelveen Gesloten", F.lit(20))
    .when(F.col("org_level_3") == "12302 - Kliniek Amstelveen Open", F.lit(20))
    .when(F.col("org_level_3") == "7251 - Kliniek gesloten Amsterdam HIC", F.lit(21))
    .when(F.col("org_level_3") == "11303 - Kliniek Gesloten Haarlem HIC", F.lit(21))
    .when(F.col("org_level_3") == "11405 - Vervolgkliniek 1 Bennebroek", F.lit(20))
    .when(F.col("org_level_3") == "11101 - Kliniek gesloten ouderen Haarlem", F.lit(34))
    .when(F.col("org_level_3") == "11430 - Vervolgkliniek 3 Bennebroek", F.lit(27))
    .when((F.col("org_level_3") == "11408 - Vervolgkliniek gesloten Bennebroek") & (F.col("rapportagedatum") <= "2024-12-31"), F.lit(0))
    .when((F.col("org_level_3") == "11408 - Vervolgkliniek gesloten Bennebroek") & (F.col("rapportagedatum") >= "2025-02-01") & (F.col("rapportagedatum") <= "2025-03-31"), F.lit(10))
    .when((F.col("org_level_3") == "11408 - Vervolgkliniek gesloten Bennebroek") & (F.col("rapportagedatum") >= "2025-04-01") & (F.col("rapportagedatum") <= "2025-06-30"), F.lit(15))
    .when((F.col("org_level_3") == "11408 - Vervolgkliniek gesloten Bennebroek") & (F.col("rapportagedatum") >= "2025-07-01"), F.lit(20))
    .otherwise(F.lit(0))
)

df_ozp = ozp_zvw.select(
    F.col("rapportagedatum"),
    F.col("beroepscategorie"),
    F.col("medewerkerberoep"),
    F.col("org_level_1"),
    F.col("org_level_2"),
    F.col("org_level_3"),
    F.col("tijd_client_direct_minuten"),
    F.col("declaratiecode_categorie"),
    F.col("financier"),
    F.col("meer_dan_365_dagen_zvw_verblijf"),
    F.col("tarief_prijspeil_2026"),
    F.col("client_code")
)

# FTE data
fte_behandeling = fte.filter(F.col("functie_groep") == "Behandeling")

# worked hours
hr_stamgegevens = fte.select(
    "personeelsnummer",
    "rapportagedatum",
    "functie",
    "functie_groep", 
    "beroepscategorie",
    "beroep",
    "org_level_3",
)

hr_stamgegevens_uniek = hr_stamgegevens.dropDuplicates(["personeelsnummer", "rapportagedatum", "org_level_3"])

diensten_verrijkt = diensten.join(
    hr_stamgegevens_uniek,
    on=["personeelsnummer", "rapportagedatum", "org_level_3"],
    how="left"
)

diensten_behandeling = diensten_verrijkt.filter(F.col("functie_groep") == "Behandeling")
diensten_behandeling.dropDuplicates(["personeelsnummer", "rapportagedatum", "org_level_3"])

geldige_diensten = [
    "Meeruren BETAALD",
    "Overwerk BETAALD",
    "Werk t Crisis-TG",
    "Meeruren TIJD",
    "Gewerkte uren",
    "Werk t AW-TG",
    "Werk t AW-TT",
    "Werk t Crisis-TT",
    "Werk t Crisis-GG",
    "Overwerk TIJD",
    "Werk t AW-GG"
]

ortec_data = diensten_behandeling.filter(F.col("werkzame_uren") == "true").filter(F.col("dienst_hoofdsoort").isin(geldige_diensten))

# Fill certain columns with certain values
fill_values = {
    "beroepscategorie": "N.v.t.",
    "medewerkerberoep": "N.v.t",
    "groepsgrootte": 0,
    "setting": "N.v.t.",
    "consult_type": "N.v.t.",
    "diagnose": "Onbekend",
    "diagnosegroep": "Onbekend",
    "tijd_client_direct_minuten": 0
}

# Combine datasets
df_combined = df_act.unionByName(df_verblijf, allowMissingColumns=True)
df_combined = df_combined.unionByName(df_ozp, allowMissingColumns=True)

df_combined = df_combined.fillna(fill_values)

# Add bruto_fte, netto_fte and fte_inzet from other dataset and join them on rapportagedatum, beroep and org_level_3. We group the data so that multiple workers will be grouped to one line
df_fte_grouped = fte_behandeling.groupBy(
    F.col("rapportagedatum"),
    F.col("org_level_3"),
    F.col("beroep").alias("medewerkerberoep")
).agg(
    F.sum("fte_bruto_contract").alias("fte_bruto_contract"),
    F.sum("fte_netto_contract").alias("fte_netto_contract"),
    F.sum("fte_inzet").alias("fte_inzet")
)

join_keys = [
    "rapportagedatum", 
    "org_level_3",
    "medewerkerberoep",
]

# Add worked hours from ortec_data
df_ortec_grouped = ortec_data.groupBy(
    F.col("rapportagedatum"),
    F.col("org_level_3"),
    F.col("beroep").alias("medewerkerberoep")
).agg(
    F.sum("dienst_uren").alias("dienst_uren")
)

# Join on final dataset
df_final_spark = df_combined.join(df_fte_grouped, on=join_keys, how="left")
df_final_spark = df_final_spark.join(df_ortec_grouped, on=join_keys, how="left")
df_final_spark = df_final_spark.fillna({
    "fte_bruto_contract": 0,
    "fte_netto_contract": 0,
    "fte_inzet": 0,
    "dienst_uren": 0,
    "available_beds": 0
})

display(df_final_spark)

In [0]:
a = df_act.filter(F.year(F.col("rapportagedatum")) == 2025).agg(F.sum("tarief_prijspeil_2026").alias("totaal_som_2025")).show()
b = df_ozp.filter(F.year(F.col("rapportagedatum")) == 2025).agg(F.sum("tarief_prijspeil_2026").alias("totaal_som_2025")).show()
c = df_verblijf.filter(F.year(F.col("rapportagedatum")) == 2025).agg(F.sum("tarief_prijspeil_2026").alias("totaal_som_2025")).show()

df_combined.filter(F.year(F.col("rapportagedatum")) == 2025).agg(F.sum("tarief_prijspeil_2026").alias("totaal_som_2025")).show()
df_final_spark.filter(F.year(F.col("rapportagedatum")) == 2025).agg(F.sum("tarief_prijspeil_2026").alias("totaal_som_2025")).show()

### Aggregate the dataset and create pandas dataframe

In [0]:
# Group the dataset per day and variable
group_columns = [
    "rapportagedatum",
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "beroepscategorie",
    "medewerkerberoep", 
    "declaratiecode_categorie", 
    "setting", 
    "groepsgrootte", 
    "consult_type"
]

# Aggregate the values
df_ml_spark = df_final_spark.groupBy(*group_columns).agg(
    F.round(F.sum("tarief_prijspeil_2026"), 2).alias("totale_omzet"),
    F.round(F.sum(F.col("tijd_client_direct_minuten").cast("double")) / 60, 2).alias("totale_uren_direct"),
    F.max(F.col("fte_bruto_contract").cast("double")).alias("fte_bruto_contract"),
    F.max(F.col("fte_netto_contract").cast("double")).alias("fte_netto_contract"),
    F.max(F.col("fte_inzet").cast("double")).alias("fte_inzet"),
    F.count("*").alias("aantal_declaraties"),
    F.countDistinct("client_code").alias("aantal_unieke_clienten"),
    F.max(F.col("available_beds")).alias("available_beds"),
    F.sum(F.col("dienst_uren").cast("double")).alias("dienst_uren")
)

# Create pandas df
df_ml_pandas = df_ml_spark.toPandas()

# Sorting
df_ml_pandas = df_ml_pandas.sort_values(by=["org_level_3", "beroepscategorie", "rapportagedatum"])
df_ml_pandas = df_ml_pandas.reset_index(drop=True)

df_ml_pandas.info()

In [0]:
cols = df_ml_spark.columns

categorische_correlatie_matrix(df_ml_spark, cols)

### Convert to right datatypes and filter to date where we have fte's

In [0]:
# to datetime
df_ml_pandas['rapportagedatum'] = pd.to_datetime(df_ml_pandas['rapportagedatum'])

# Filter rapportagedatum > 2023
df_ml_pandas = df_ml_pandas[df_ml_pandas['rapportagedatum'] > '2022-12-31']
df_ml_pandas = df_ml_pandas.reset_index(drop=True)

df_ml_pandas.head(5)

### Feature engineering

In [0]:
# Function for calendar features
def calendar_features(df_raw, date_col='rapportagedatum'):
    df = df_raw.copy()
    
    # Convert to datetime
    df[date_col] = pd.to_datetime(df[date_col])
    
    df['weekday']     = df[date_col].dt.dayofweek
    df['month']       = df[date_col].dt.month
    df['week']        = df[date_col].dt.isocalendar().week.astype(int)
    df['year']        = df[date_col].dt.year
    df['quarter']     = df[date_col].dt.quarter
    df['day_of_year'] = df[date_col].dt.dayofyear

    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    
    df['is_weekend']  = (df['weekday'] >= 5).astype(int)

    # Holidays
    vakanties_noord = [
        ('voorjaar', '2022-02-19', '2022-02-27'), ('mei', '2022-04-23', '2022-05-08'), ('zomer', '2022-07-09', '2022-08-21'), ('herfst', '2022-10-15', '2022-10-23'), ('kerst', '2022-12-24', '2023-01-08'),
        ('voorjaar', '2023-02-25', '2023-03-05'), ('mei', '2023-04-22', '2023-04-30'), ('zomer', '2023-07-08', '2023-08-20'), ('herfst', '2023-10-14', '2023-10-22'), ('kerst', '2023-12-23', '2024-01-07'),
        ('voorjaar', '2024-02-17', '2024-02-25'), ('mei', '2024-04-27', '2024-05-05'), ('zomer', '2024-07-20', '2024-09-01'), ('herfst', '2024-10-19', '2024-10-27'), ('kerst', '2024-12-21', '2025-01-05'),
        ('voorjaar', '2025-02-22', '2025-03-02'), ('mei', '2025-04-19', '2025-04-27'), ('zomer', '2025-07-19', '2025-08-31'), ('herfst', '2025-10-18', '2025-10-26'), ('kerst', '2025-12-20', '2026-01-04'),
    ]
    for vtype in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']:
        df[f'is_{vtype}vakantie'] = 0
        
    for vtype, start, end in vakanties_noord:
        mask = (df[date_col] >= pd.to_datetime(start)) & (df[date_col] <= pd.to_datetime(end))
        df.loc[mask, f'is_{vtype}vakantie'] = 1
        
    vakantie_cols = [f'is_{v}vakantie' for v in ['voorjaar', 'mei', 'zomer', 'herfst', 'kerst']]
    df['is_schoolvakantie'] = (df[vakantie_cols].sum(axis=1) > 0).astype(int)

    feestdagen = pd.to_datetime([
        '2022-01-01', '2022-04-15', '2022-04-17', '2022-04-18', '2022-04-27', '2022-05-05', '2022-05-26', '2022-06-05', '2022-06-06', '2022-12-25', '2022-12-26',
        '2023-01-01', '2023-04-07', '2023-04-09', '2023-04-10', '2023-04-27', '2023-05-05', '2023-05-18', '2023-05-28', '2023-05-29', '2023-12-25', '2023-12-26',
        '2024-01-01', '2024-03-29', '2024-03-31', '2024-04-01', '2024-04-27', '2024-05-05', '2024-05-09', '2024-05-19', '2024-05-20', '2024-12-25', '2024-12-26',
        '2025-01-01', '2025-04-18', '2025-04-20', '2025-04-21', '2025-04-26', '2025-05-05', '2025-05-29', '2025-06-08', '2025-06-09', '2025-12-25', '2025-12-26',
    ])
    df['is_feestdag'] = df[date_col].isin(feestdagen).astype(int)

    return df

In [0]:
# Add calendar features
df_calendar = calendar_features(df_ml_pandas)

### One hot encoding and adding lags and MA's

In [0]:
# Deze cel niet gebruiken, zorgt voor data leakage
cat_cols = [
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "beroepscategorie", 
    "medewerkerberoep", 
    "declaratiecode_categorie", 
    "setting", 
    "consult_type"
]

df_calendar['groepsgrootte'] = df_calendar['groepsgrootte'].astype(str)
cat_cols.append("groepsgrootte")

# Add time series features
df_ts = df_calendar.sort_values(cat_cols + ['rapportagedatum']).copy()

# Absolute time index for long term trend
df_ts['time_index'] = (df_ts['rapportagedatum'] - df_ts['rapportagedatum'].min()).dt.days

target_cols = ['totale_omzet', 'totale_uren_direct', 'aantal_unieke_clienten', 'aantal_declaraties']

for target in target_cols:
    grouped = df_ts.groupby(cat_cols)[target]
    
    # Calculate lags per target
    for lag in [1, 7]:
        df_ts[f'{target}_lag_{lag}d'] = grouped.shift(lag)
        
    # Calculate MA's per target
    for window in [7, 28]:
        df_ts[f'{target}_rolling_mean_{window}d'] = grouped.transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
        
    # SARIMAX trend difference
    df_ts[f'{target}_trend_verschil_7d'] = df_ts[f'{target}_lag_1d'] - df_ts[f'{target}_lag_7d']

# Drop NaN rows which exist due to the rolling_mean_28d
df_ts = df_ts.dropna().reset_index(drop=True)

In [0]:
# Calculate KPI's per FTE
tijdstappen = ['_lag_1d', '_lag_7d', '_rolling_mean_7d', '_rolling_mean_28d']

for stap in tijdstappen:
    uren     = f"totale_uren_direct{stap}"
    clienten = f"aantal_unieke_clienten{stap}"
    omzet    = f"totale_omzet{stap}"
    
    # FTE is nu een harde input per rij, dus we halen {stap} hier weg!
    fte_inzet = "fte_inzet"
    fte_bruto = "fte_bruto_contract"
    fte_netto = "fte_netto_contract"

    # 1. Algemene KPI's (deze gebruiken wél beide de historie)
    kpi_prijs   = f"gem_prijs{stap}"
    kpi_uren_cl = f"kpi_uren_per_client{stap}"

    df_ts[kpi_prijs]   = np.where(df_ts[uren] > 0, df_ts[omzet] / df_ts[uren], 0)
    df_ts[kpi_uren_cl] = np.where(df_ts[clienten] > 0, df_ts[uren] / df_ts[clienten], 0)

    # 2. KPI's for fte_inzet (Historische productie gedeeld door huidige capaciteit)
    kpi_uren_fte_inzet = f"kpi_uren_per_fte_inzet{stap}"
    kpi_cl_fte_inzet   = f"kpi_clienten_per_fte_inzet{stap}"
    
    df_ts[kpi_uren_fte_inzet] = np.where(df_ts[fte_inzet] > 0, df_ts[uren] / df_ts[fte_inzet], 0)
    df_ts[kpi_cl_fte_inzet]   = np.where(df_ts[fte_inzet] > 0, df_ts[clienten] / df_ts[fte_inzet], 0)

    # 3. KPI's for fte_bruto
    kpi_uren_fte_bruto = f"kpi_uren_per_fte_bruto{stap}"
    kpi_cl_fte_bruto   = f"kpi_clienten_per_fte_bruto{stap}"
    
    df_ts[kpi_uren_fte_bruto] = np.where(df_ts[fte_bruto] > 0, df_ts[uren] / df_ts[fte_bruto], 0)
    df_ts[kpi_cl_fte_bruto]   = np.where(df_ts[fte_bruto] > 0, df_ts[clienten] / df_ts[fte_bruto], 0)

    # 4. KPI's for fte_netto
    kpi_uren_fte_netto = f"kpi_uren_per_fte_netto{stap}"
    kpi_cl_fte_netto   = f"kpi_clienten_per_fte_netto{stap}"
    
    df_ts[kpi_uren_fte_netto] = np.where(df_ts[fte_netto] > 0, df_ts[uren] / df_ts[fte_netto], 0)
    df_ts[kpi_cl_fte_netto]   = np.where(df_ts[fte_netto] > 0, df_ts[clienten] / df_ts[fte_netto], 0)

In [0]:
df_calendar.columns

In [0]:
# One hot encoding
cat_cols = [
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "beroepscategorie", 
    "medewerkerberoep", 
    "declaratiecode_categorie", 
    "setting", 
    "consult_type"
]

df_calendar['groepsgrootte'] = df_calendar['groepsgrootte'].astype(str)
cat_cols.append("groepsgrootte")

df_ohe = pd.get_dummies(df_calendar, columns=cat_cols, drop_first=False)

bool_cols = df_ohe.select_dtypes(include=['bool', 'uint8']).columns
df_ohe[bool_cols] = df_ohe[bool_cols].astype(int)

In [0]:
df_ohe.head(5)

### Train test split

Starting with splitting the model in the train_test sets

In [0]:
# Sorting the values on date
df_model = df_ohe.sort_values('rapportagedatum').copy()

verboden_kolommen = ['rapportagedatum', 'totale_omzet', 'totale_uren_direct', 'aantal_declaraties', 'aantal_unieke_clienten', 'time_index']
features = [col for col in df_model.columns if col not in verboden_kolommen]

df_model['year'] = df_model['year'].astype(int)
train_mask = df_model["year"].isin([2023, 2024])
test_mask = df_model["year"] == 2025

# Train test split
X_train, y_train = df_model.loc[train_mask, features], df_model.loc[train_mask, 'totale_omzet']
X_test, y_test = df_model.loc[test_mask, features], df_model.loc[test_mask, 'totale_omzet']

dates_train = df_model.loc[train_mask, 'rapportagedatum']
dates_test = df_model.loc[test_mask, 'rapportagedatum']

tscv = TimeSeriesSplit(n_splits=4)

In [0]:
pad_dataset = './dataset'
os.makedirs(pad_dataset, exist_ok=True)

# Save dataset
X_train.to_csv(f'{pad_dataset}/X_train.csv', index=False)
X_test.to_csv(f'{pad_dataset}/X_test.csv', index=False)
y_train.to_csv(f'{pad_dataset}/y_train.csv', index=False)
y_test.to_csv(f'{pad_dataset}/y_test.csv', index=False)
dates_train.to_csv(f'{pad_dataset}/dates_train.csv', index=False)
dates_test.to_csv(f'{pad_dataset}/dates_test.csv', index=False)

print(f"✅ Dataset (Train, Test & Dates) succesvol opgeslagen in de map: {pad_dataset}")

### Training XGBoost model

In [0]:
# Model
param_dist = {
    'n_estimators': [100, 300, 500, 800],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],         # Hoeveel % van de rijen gebruikt hij per boom?
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],  # Hoeveel % van de features gebruikt hij per boom?
    'reg_alpha': [0, 0.1, 1.0, 5.0],           # L1 Regularisatie
    'reg_lambda': [0.1, 1.0, 5.0, 10.0]        # L2 Regularisatie
}

# base_model
xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# random search for hyperparameters
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,                                 
    scoring='neg_mean_absolute_error',
    cv=tscv,
    verbose=1,                                 
    random_state=42,
    n_jobs=-1
)

# start model training
random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

# feature importance
feature_importances = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 15 Belangrijkste Features ---")
print(feature_importances.head(15))

In [0]:
# Save XGBoost model
pad_xgb = './xgb_resultaten'
os.makedirs(pad_xgb, exist_ok=True)

test_evaluatie = pd.DataFrame({
    'rapportagedatum': dates_test,
    'totale_omzet': y_test,
    'Voorspelde_Omzet': y_pred
})

# save dataframes
test_evaluatie.to_csv(f'{pad_xgb}/test_results.csv', index=False)
X_train.to_csv(f'{pad_xgb}/X_train.csv', index=False)
X_test.to_csv(f'{pad_xgb}/X_test.csv', index=False)

# Save model
best_model.save_model(f'{pad_xgb}/getraind_xgboost_model.json')

print(f"✅ XGBoost model, voorspellingen en tabellen zijn succesvol opgeslagen in de map: {pad_xgb}")

### RF
Building a random forest model for testing another ensemble method

In [0]:
param_dist_rf = {
    'n_estimators': [100, 200],                     # Minder bomen per bos
    'max_depth': [10, 15],                          # Strenge limiet op de diepte
    'min_samples_split': [10, 20],               
    'min_samples_leaf': [4, 10],                 
    'max_features': ['sqrt'],                       # Alleen een sub-sample van features per boom
    'bootstrap': [True]                             
}

# base model
rf_base = RandomForestRegressor(random_state=42, n_jobs=2)

# random search
random_search_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=10,                                      # Minder iteraties (was 20)
    scoring='neg_mean_absolute_error',
    cv=tscv,  
    verbose=2,                                      # Verbose=2 laat je precies zien in de logs hoe ver hij is
    random_state=42,
    n_jobs=2                                        # Beperk het kopiëren van data in het RAM
)

# train model
random_search_rf.fit(X_train, y_train)
best_model_rf = random_search_rf.best_estimator_

# Predict on test set
y_pred_rf = best_model_rf.predict(X_test)

In [0]:
# Save RF model
pad_rf = './rf_resultaten'
os.makedirs(pad_rf, exist_ok=True)

test_evaluatie = pd.DataFrame({
    'rapportagedatum': dates_test,
    'totale_omzet': y_test,
    'Voorspelde_Omzet': y_pred_rf
})

# save dataframes
test_evaluatie.to_csv(f'{pad_rf}/test_results.csv', index=False)
X_train.to_csv(f'{pad_rf}/X_train.csv', index=False)
X_test.to_csv(f'{pad_rf}/X_test.csv', index=False)

# SAVE MODEL (Aangepast voor scikit-learn!)
joblib.dump(best_model_rf, f'{pad_rf}/getraind_rf_model.pkl')

print(f"✅ Random Forest model, voorspellingen en tabellen zijn succesvol opgeslagen in de map: {pad_rf}")